# Dashboard Figures — Fleet Health Telemetry

**Purpose**: Prototype the 2-page Dash+Plotly dashboard for fleet health monitoring.  
**Data Source**: Golden layer outputs from `src/pipeline.py`  
**Author**: Patricio Ortiz  
**Date**: June 2026

---

## Dashboard Structure

| Page | Question Answered | Key Widgets |
|------|-------------------|-------------|
| **Fleet Overview** | How is my fleet behaving currently? | Status donut, heatmap, priority table, AI summaries |
| **Unit Detail** | What data backs the conclusions? | Signal time series, KPI tables, per-system drill-down |

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# --- Data paths ---
GOLDEN_PATH = Path('../data/telemetry/golden/cda')
SILVER_PATH = Path('../data/telemetry/silver/cda/Telemetry_Wide_With_States')

# --- Load Golden layer results ---
unit_health = pd.read_parquet(GOLDEN_PATH / 'unit_health/year=2026/week=24/unit_health.parquet')
system_health = pd.read_parquet(GOLDEN_PATH / 'system_health/year=2026/week=24/system_health.parquet')
deviation = pd.read_parquet(GOLDEN_PATH / 'technique_results/deviation/year=2026/week=24/deviation_results.parquet')
events = pd.read_parquet(GOLDEN_PATH / 'technique_results/events/year=2026/week=24/events.parquet')
trends = pd.read_parquet(GOLDEN_PATH / 'technique_results/trend/year=2026/week=24/trend_results.parquet')

# --- Load raw telemetry for time series plots (90 days ≈ 13 weeks) ---
raw_files = sorted(SILVER_PATH.glob('*.parquet'))[-13:]  # Last 13 weeks (~90 days)
raw_df = pd.concat([pd.read_parquet(f) for f in raw_files], ignore_index=True)
raw_df['Fecha'] = pd.to_datetime(raw_df['Fecha'])

# --- Load baselines for limit lines ---
import yaml
signal_registry = yaml.safe_load(open('../data/telemetry/config/cda/signal_registry.yaml'))
baselines_path = Path('../data/telemetry/silver/cda/baselines')
baseline = pd.read_parquet(sorted(baselines_path.glob('baseline_*.parquet'))[-1])

# --- Color palette ---
STATUS_COLORS = {'Normal': '#2ecc71', 'Alerta': '#f39c12', 'Anormal': '#e74c3c', 'InsufficientData': '#95a5a6'}

print(f'Units: {unit_health.unit.nunique()} | Systems: {system_health.system.nunique()}')
print(f'Events: {len(events):,} | Trends: {len(trends)} | Raw rows: {len(raw_df):,}')

Units: 10 | Systems: 4
Events: 708,766 | Trends: 594 | Raw rows: 269,486


---

# PAGE 1: Fleet Overview

**Question**: *"How is my fleet behaving currently?"*

Shows: status distribution, system heatmap, unit prioritization, and AI-generated explanations.

In [2]:
# Figure 1: Fleet Status Distribution (donut)
status_counts = unit_health['overall_status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']

fig_donut = px.pie(
    status_counts, values='Count', names='Status',
    color='Status', color_discrete_map=STATUS_COLORS,
    hole=0.5, title='Fleet Health Status'
)
fig_donut.update_layout(width=350, height=300, margin=dict(t=40, b=20))
fig_donut.show()

In [3]:
# Figure 2: System Health Heatmap (units × systems)
pivot = system_health.pivot(index='unit', columns='system', values='system_score')
# Sort units by overall priority
unit_order = unit_health.sort_values('priority_score', ascending=False)['unit'].tolist()
pivot = pivot.reindex(unit_order)

fig_heatmap = px.imshow(
    pivot,
    color_continuous_scale=['#2ecc71', '#f39c12', '#e74c3c'],
    zmin=0, zmax=100,
    labels=dict(color='Risk Score'),
    title='System Health by Unit (0=Healthy, 100=Critical)',
    text_auto='.0f'
)
fig_heatmap.update_layout(height=380, width=650)
fig_heatmap.show()

In [4]:
# Figure 3: Unit Priority Table (sorted, with status and key info)
priority_table = unit_health.sort_values('priority_score', ascending=False)[[
    'unit', 'overall_status', 'priority_score', 'unit_score', 
    'n_anormal_systems', 'n_alerta_systems', 'top_risk_systems'
]].copy()
priority_table.columns = ['Unit', 'Status', 'Priority', 'Score', 'Anormal Systems', 'Alerta Systems', 'Top Risk']

# Color rows by status
fill_colors = [STATUS_COLORS.get(s, '#ffffff') + '40' for s in priority_table['Status']]

fig_priority_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(priority_table.columns),
        fill_color='#2c3e50', font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[priority_table[c] for c in priority_table.columns],
        align='center', height=28,
        font=dict(size=11)
    )
)])
fig_priority_table.update_layout(title='Fleet Priority Ranking', height=350, width=850, margin=dict(t=40, b=10))
fig_priority_table.show()

In [5]:
# Figure 4: AI Explanation Table (simulated — would come from LLM in production)
# In production, this reads from unit_health['executive_summary'] column
# For prototype, we generate a placeholder based on available data

explanations = []
for _, row in unit_health.sort_values('priority_score', ascending=False).iterrows():
    unit = row['unit']
    status = row['overall_status']
    top_systems = row['top_risk_systems']
    
    if status == 'Alerta':
        msg = f"{unit} requires attention. Systems at risk: {', '.join(eval(top_systems) if isinstance(top_systems, str) else top_systems[:2])}."
    elif status == 'Anormal':
        msg = f"{unit} CRITICAL — immediate inspection needed. Affected: {', '.join(eval(top_systems) if isinstance(top_systems, str) else top_systems[:2])}."
    else:
        msg = f"{unit} operating within normal parameters."
    explanations.append({'Unit': unit, 'Status': status, 'Assessment': msg})

expl_df = pd.DataFrame(explanations)

fig_explanations = go.Figure(data=[go.Table(
    header=dict(
        values=['Unit', 'Status', 'AI Assessment'],
        fill_color='#34495e', font=dict(color='white', size=12),
        align='left'
    ),
    cells=dict(
        values=[expl_df['Unit'], expl_df['Status'], expl_df['Assessment']],
        align='left', height=30,
        font=dict(size=11)
    )
)])
fig_explanations.update_layout(title='Fleet Health Assessment (AI-Generated)', height=380, width=900, margin=dict(t=40, b=10))
fig_explanations.show()

---

# PAGE 2: Unit Detail

**Question**: *"What data backs the conclusions we are presenting?"*

Layout:
1. `[Filter: Unit]`
2. `[AI comment on unit]`
3. `[Table: system | risk — sorted by risk]`
4. `[Filter: System]`
5. `[Table: most concerning signals in system]`
6. For each signal (sorted by criticality):
   - Left: Time series (rolling mean + limits + trend line)
   - Right: KPI table (events, longest episode, trend, distribution shift)

In [6]:
# === FILTERS (simulated with variables) ===
SELECTED_UNIT = 'T_12'    # Dropdown in Dash
SELECTED_SYSTEM = 'Transmission'  # Dropdown in Dash

print(f'Selected Unit: {SELECTED_UNIT}')
print(f'Selected System: {SELECTED_SYSTEM}')

Selected Unit: T_12
Selected System: Transmission


In [7]:
# --- Section: Unit AI Comment ---
unit_row = unit_health[unit_health['unit'] == SELECTED_UNIT].iloc[0]
print(f"Unit: {SELECTED_UNIT}")
print(f"Status: {unit_row['overall_status']} | Priority Score: {unit_row['priority_score']}")
print(f"Top risk systems: {unit_row['top_risk_systems']}")
print("---")
# In production: display unit_row['executive_summary'] from LLM
print("AI Summary: This unit shows elevated risk in Transmission (lockup slip and transmission slip " 
      "trending upward) and Engine (turbo pressures drifting). Schedule inspection within 48h.")

Unit: T_12
Status: Anormal | Priority Score: 128.5
Top risk systems: ['Engine' 'Transmission' 'Steering']
---
AI Summary: This unit shows elevated risk in Transmission (lockup slip and transmission slip trending upward) and Engine (turbo pressures drifting). Schedule inspection within 48h.


In [8]:
# --- Section: System Risk Table (sorted by risk) ---
unit_systems = system_health[system_health['unit'] == SELECTED_UNIT][[
    'system', 'system_score', 'system_status', 'n_techniques_triggered'
]].sort_values('system_score', ascending=False).copy()
unit_systems.columns = ['System', 'Risk Score', 'Status', 'Techniques Triggered']

fig_sys_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(unit_systems.columns),
        fill_color='#2c3e50', font=dict(color='white', size=12), align='center'
    ),
    cells=dict(
        values=[unit_systems[c] for c in unit_systems.columns],
        align='center', height=28, font=dict(size=12)
    )
)])
fig_sys_table.update_layout(
    title=f'{SELECTED_UNIT} — System Risk Summary',
    height=220, width=600, margin=dict(t=40, b=10)
)
fig_sys_table.show()

In [9]:
# --- Section: Most Concerning Signals in Selected System ---
# Combine deviation + events + trends for signal-level overview

system_signals = deviation[
    (deviation['unit'] == SELECTED_UNIT) & (deviation['system'] == SELECTED_SYSTEM)
].sort_values('risk_score', ascending=False)

# Enrich with event count per signal
unit_events = events[events['unit'] == SELECTED_UNIT]
event_counts = unit_events.groupby('feature').agg(
    n_events=('event_group', 'count'),
    max_duration=('duration_minutes', 'max'),
    n_warnings=('event_type_weighted', lambda x: (x == 'warning').sum())
).reset_index().rename(columns={'feature': 'signal'})

signal_summary = system_signals.merge(event_counts, on='signal', how='left').fillna(0)
signal_summary = signal_summary[[
    'signal', 'risk_score', 'status', 'abnormal_pct', 'n_events', 'max_duration', 'n_warnings'
]].sort_values('risk_score', ascending=False)
signal_summary.columns = ['Signal', 'Risk Score', 'Status', 'Abnormal %', 'Events', 'Max Episode (min)', 'Warnings']

fig_signal_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(signal_summary.columns),
        fill_color='#8e44ad', font=dict(color='white', size=11), align='center'
    ),
    cells=dict(
        values=[signal_summary[c] for c in signal_summary.columns],
        align='center', height=26, font=dict(size=11),
        format=[None, '.1f', None, '.1f', '.0f', '.0f', '.0f']
    )
)])
fig_signal_table.update_layout(
    title=f'{SELECTED_UNIT} / {SELECTED_SYSTEM} — Signal Overview (sorted by risk)',
    height=250, width=850, margin=dict(t=40, b=10)
)
fig_signal_table.show()

### Signal Detail: Time Series + KPI Cards

For each signal in the selected system, we show:
- **Left**: Rolling mean time series with P5/P95 limits and linear regression trend line
- **Right**: KPI summary table (events, longest episode, trend, distribution shift)

In [10]:
def get_signal_kpis(unit, signal, events_df, trends_df):
    """Build KPI dict for a signal."""
    # Events
    sig_events = events_df[(events_df['unit'] == unit) & (events_df['feature'] == signal)]
    n_events = len(sig_events)
    max_episode = int(sig_events['duration_minutes'].max()) if n_events > 0 else 0
    n_warnings = int((sig_events['event_type_weighted'] == 'warning').sum())
    
    # Trend (use 4-week window)
    sig_trend = trends_df[
        (trends_df['unit'] == unit) & 
        (trends_df['signal'] == signal) & 
        (trends_df['window_weeks'] == 4)
    ]
    if not sig_trend.empty:
        t = sig_trend.iloc[0]
        trend_detected = 'Yes' if t['is_significant'] and t['is_good_fit'] else 'No'
        trend_formula = f"{t['slope_per_day']:+.3f}/day (R²={t['r2']:.2f})" if trend_detected == 'Yes' else '—'
        trend_direction = t['trend_interpretation'] if trend_detected == 'Yes' else '—'
    else:
        trend_detected = 'N/A'
        trend_formula = '—'
        trend_direction = '—'
    
    return {
        'Total Events': n_events,
        'Warnings': n_warnings,
        'Longest Episode': f'{max_episode} min',
        'Trend Detected': trend_detected,
        'Trend Direction': trend_direction,
        'Trend Formula': trend_formula,
    }


def plot_signal_with_kpis(unit, signal, signal_meta, raw_data, baseline_df, events_df, trends_df):
    """Create side-by-side: time series plot + KPI table for one signal."""
    
    # --- Get signal data ---
    unit_data = raw_data[raw_data['Unit'] == unit][['Fecha', signal, 'Estado']].dropna(subset=[signal])
    unit_data = unit_data.sort_values('Fecha')
    
    if unit_data.empty:
        return None
    
    # Rolling mean (30 min)
    unit_data['rolling_mean'] = unit_data[signal].rolling(30, min_periods=1).mean()
    
    # --- Create subplot: left=timeseries, right=KPI table ---
    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.7, 0.3],
        specs=[[{'type': 'scatter'}, {'type': 'table'}]]
    )
    
    # --- Left: Time series ---
    fig.add_trace(
        go.Scatter(
            x=unit_data['Fecha'], y=unit_data['rolling_mean'],
            mode='lines', name='Rolling Mean (30min)',
            line=dict(color='#2c3e50', width=1.5)
        ), row=1, col=1
    )
    
    # Add P95/P5 limit lines from baseline (for Operacional state)
    risk_dir = signal_meta.get('risk_direction', 'high')
    # Try to get limits from baseline
    sig_baseline = baseline_df[
        (baseline_df['signal'] == signal) & 
        (baseline_df['state'] == 'Operacional')
    ] if 'signal' in baseline_df.columns else pd.DataFrame()
    
    if not sig_baseline.empty:
        row_bl = sig_baseline.iloc[0]
        if risk_dir in ['high', 'both']:
            fig.add_hline(y=row_bl.get('P95', np.nan), line_dash='dash', line_color='orange',
                         annotation_text='P95', row=1, col=1)
            fig.add_hline(y=row_bl.get('P99', np.nan), line_dash='dash', line_color='red',
                         annotation_text='P99', row=1, col=1)
        if risk_dir in ['low', 'both']:
            fig.add_hline(y=row_bl.get('P5', np.nan), line_dash='dash', line_color='orange',
                         annotation_text='P5', row=1, col=1)
            fig.add_hline(y=row_bl.get('P1', np.nan), line_dash='dash', line_color='red',
                         annotation_text='P1', row=1, col=1)
    
    # Add trend line (linear regression)
    sig_trend = trends_df[
        (trends_df['unit'] == unit) & (trends_df['signal'] == signal) & (trends_df['window_weeks'] == 4)
    ]
    if not sig_trend.empty and sig_trend.iloc[0]['is_significant']:
        t = sig_trend.iloc[0]
        x_hours = (unit_data['Fecha'] - unit_data['Fecha'].min()).dt.total_seconds() / 3600
        slope_per_hour = t['slope_per_day'] / 24
        # Approximate: use rolling_mean start as intercept
        intercept = unit_data['rolling_mean'].iloc[0]
        trend_line = intercept + slope_per_hour * x_hours
        
        trend_color = '#e74c3c' if t['trend_interpretation'] == 'worsening' else '#2ecc71'
        fig.add_trace(
            go.Scatter(
                x=unit_data['Fecha'], y=trend_line,
                mode='lines', name=f"Trend ({t['trend_interpretation']})",
                line=dict(color=trend_color, width=2, dash='dot')
            ), row=1, col=1
        )
    
    # --- Right: KPI Table ---
    kpis = get_signal_kpis(unit, signal, events_df, trends_df)
    kpi_names = list(kpis.keys())
    kpi_values = [str(v) for v in kpis.values()]
    
    fig.add_trace(
        go.Table(
            header=dict(
                values=['Metric', 'Value'],
                fill_color='#34495e', font=dict(color='white', size=11),
                align='left'
            ),
            cells=dict(
                values=[kpi_names, kpi_values],
                align='left', height=25, font=dict(size=11)
            )
        ), row=1, col=2
    )
    
    # Layout
    display_name = signal_meta.get('display_name', signal)
    fig.update_layout(
        title=f'{signal} — {display_name}',
        height=300, width=1000,
        showlegend=True,
        margin=dict(t=40, b=30, l=50, r=20)
    )
    fig.update_xaxes(title_text='Time', row=1, col=1)
    fig.update_yaxes(title_text=signal_meta.get('unit', ''), row=1, col=1)
    
    return fig

In [11]:
# --- Render signal detail for each signal in selected system ---

# Get signals in selected system (sorted by risk score)
system_signal_list = deviation[
    (deviation['unit'] == SELECTED_UNIT) & (deviation['system'] == SELECTED_SYSTEM)
].sort_values('risk_score', ascending=False)['signal'].tolist()

# Build metadata lookup
signal_meta_map = {s['name']: s for s in signal_registry['signals']}

print(f'Rendering {len(system_signal_list)} signals for {SELECTED_UNIT} / {SELECTED_SYSTEM}:')
print(f'  Signals: {system_signal_list}')
print()

for signal in system_signal_list:
    meta = signal_meta_map.get(signal, {'name': signal, 'display_name': signal, 'unit': '', 'risk_direction': 'high'})
    fig = plot_signal_with_kpis(
        SELECTED_UNIT, signal, meta, raw_df, baseline, events, trends
    )
    if fig:
        fig.show()
    else:
        print(f'  ⚠️ No data for {signal}')

Rendering 5 signals for T_12 / Transmission:
  Signals: ['TrnSlip', 'DiffTemp', 'DiffLubePres', 'TrnLubeTemp', 'LckupSlip']



---

## Summary — Dashboard Layout

### Page 1: Fleet Overview
```
┌───────────────────────────────────────────────────────────────┐
│  [Donut: Status]  │  [Heatmap: Units × Systems]              │
├───────────────────┴──────────────────────────────────────────┤
│  [Table: Unit Priority — sorted by priority_score]           │
├──────────────────────────────────────────────────────────────┤
│  [Table: AI Assessment per unit]                             │
└──────────────────────────────────────────────────────────────┘
```

### Page 2: Unit Detail
```
┌──────────────────────────────────────────────────────────────┐
│  [Filter: Unit ▼]                                            │
├──────────────────────────────────────────────────────────────┤
│  [AI Comment on selected unit]                               │
├──────────────────────────────────────────────────────────────┤
│  [Table: System | Risk Score | Status — sorted by risk]      │
├──────────────────────────────────────────────────────────────┤
│  [Filter: System ▼]                                          │
├──────────────────────────────────────────────────────────────┤
│  [Table: Signal overview — sorted by criticality]            │
├──────────────────────────────────────────────────────────────┤
│  For each signal in system:                                  │
│  ┌────────────────────────────┬────────────────────────────┐ │
│  │  [Time Series Plot]        │  [KPI Table]               │ │
│  │  • Rolling mean (30min)    │  • Total Events            │ │
│  │  • P95/P99 limit lines     │  • Warnings count          │ │
│  │  • Trend regression line   │  • Longest Episode         │ │
│  │                            │  • Trend Detected?         │ │
│  │                            │  • Trend Direction          │ │
│  │                            │  • Trend Formula           │ │
│  └────────────────────────────┴────────────────────────────┘ │
│  (repeated per signal, sorted by risk)                       │
└──────────────────────────────────────────────────────────────┘
```